# 기계 고장 예측 분류 모델 (예지보전) — 외부 데이터 교차 검증판
**학번_이름** | 모델: **분류** | 주 데이터: 수업 예시 〈기계 고장〉(AI4I 2020, 10,000행 × 14열)

| 구분 | 내용 |
|---|---|
| 예측할 것 | 공정 센서 값을 보고 기계가 **고장(1)인지 정상(0)인지** 분류 |
| 진로 연결 | 산업공학 – 설비보전(TPM)·예지보전, 비용을 고려한 의사결정 |
| 기계학습인 이유 | 고장을 바로 계산하는 공식이 없음. 여러 센서 값의 **조합**에서 패턴을 배워야 함 |
| 검증 전략 | ① 교차 검증으로 모델 고르기 ② 평가용 데이터는 마지막에 한 번만 사용 ③ **성격이 다른 외부 데이터 2개**(NASA 항공기 엔진, Microsoft 공장 설비)에 같은 방법을 적용해 결론이 유지되는지 확인 |

**목차**: 0 준비 → 1 데이터 확인 → 2 전처리·특징 만들기 → 3 모델 비교(교차 검증) → 4 비용 기반 판단 기준 → 5 최종 평가 → 6 해석 → 7 외부 검증 A: NASA 엔진 → 8 외부 검증 B: Microsoft 설비 → 9 종합 결론

## 0. 준비

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, confusion_matrix, precision_recall_curve)
SEED = 42
np.random.seed(SEED)

# 고장을 놓쳤을 때 손해(FN)가 괜히 점검했을 때 손해(FP)보다 몇 배인지 (가정, 4장에서 바꿔 보며 확인)
COST_FN, COST_FP = 10, 1

def scores(y, pred, prob=None):
    r = {'정확도': accuracy_score(y, pred), '정밀도': precision_score(y, pred, zero_division=0),
         '재현율': recall_score(y, pred), 'F1': f1_score(y, pred)}
    if prob is not None: r['PR-AUC'] = average_precision_score(y, prob)
    return {k: round(v, 3) for k, v in r.items()}

def cost(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return fn*COST_FN + fp*COST_FP

def best_threshold(y, prob):
    ts = np.round(np.arange(0.05, 0.96, 0.01), 2)
    cs = [cost(y, (prob >= t).astype(int)) for t in ts]
    return ts[int(np.argmin(cs))], ts, cs

def plot_cm(y, pred, title):
    cm = confusion_matrix(y, pred, labels=[0,1])
    plt.figure(figsize=(4,3.4)); plt.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i,j], ha='center', va='center', fontsize=15,
                     color='white' if cm[i,j] > cm.max()/2 else 'black')
    plt.xticks([0,1], ['pred normal','pred failure']); plt.yticks([0,1], ['real normal','real failure'])
    plt.title(title); plt.tight_layout(); plt.show()
    tn, fp, fn, tp = cm.ravel()
    print(f'실제 고장 {tp+fn}건 중 {tp}건을 미리 잡음({tp/(tp+fn)*100:.1f}%), {fn}건 놓침 / 헛경보 {fp}건')

## 1. 데이터 확인
수업 화면에서 받은 〈기계 고장〉 CSV를 왼쪽 📁에 올리고 `DATA_PATH`를 그 파일 이름으로 바꿔요.

In [ ]:
DATA_PATH = '학번_이름_데이터.csv'   # ← 내 파일 이름

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df.columns = df.columns.str.strip()
print('크기:', df.shape)
display(df.head())
df.info()

In [ ]:
print('빈 값:', int(df.isnull().sum().sum()), '/ 중복 행:', int(df.duplicated().sum()))
n_fail = int(df['Machine failure'].sum())
print(f"고장 {n_fail}건 / 전체 {len(df)}건 → 고장 비율 {n_fail/len(df)*100:.1f}%")
print('고장 종류별 건수:', df[['TWF','HDF','PWF','OSF','RNF']].sum().to_dict())
print('제품 등급:', df['Type'].value_counts().to_dict())

➡️ 고장이 약 3.4%뿐인 **불균형 데이터**예요. 무조건 '정상'이라고만 해도 정확도가 96.6%가 나와요.
그래서 이 탐구에서는 **정확도 대신 재현율·정밀도·F1·PR-AUC**(정밀도-재현율 곡선 아래 넓이, 1에 가까울수록 좋음)로 평가해요.

In [ ]:
num_cols = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6))
for ax, c in zip(axes, num_cols):
    ax.boxplot([df.loc[df['Machine failure']==k, c] for k in [0,1]])
    ax.set_xticks([1,2]); ax.set_xticklabels(['normal','failure']); ax.set_title(c)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4.5))
ok = df['Machine failure']==0
plt.scatter(df.loc[ok,'Rotational speed [rpm]'], df.loc[ok,'Torque [Nm]'], s=3, alpha=.25, label='normal')
plt.scatter(df.loc[~ok,'Rotational speed [rpm]'], df.loc[~ok,'Torque [Nm]'], s=10, c='red', label='failure')
plt.xlabel('Rotational speed [rpm]'); plt.ylabel('Torque [Nm]'); plt.legend(); plt.title('Speed vs Torque'); plt.show()

➡️ 고장은 **토크가 크고 속도가 낮은 쪽**과 **토크가 작고 속도가 높은 쪽**, 양 끝에 몰려 있어요.
한 값만으로 선 하나를 그어 나눌 수 없어서, 여러 값을 함께 보는 모델이 필요해요. 이 둘을 곱하면 **동력**이 되므로 동력을 새 특징으로 만들 근거가 돼요.

## 2. 전처리와 특징 만들기
| 처리 | 이유 |
|---|---|
| `UDI`, `Product ID` 빼기 | 번호·이름표라 고장과 관계없음 |
| `TWF, HDF, PWF, OSF, RNF` 빼기 | 고장이 난 **뒤에** 알 수 있는 '고장 종류'. 넣으면 답을 보고 푸는 셈(**데이터 누수**). 6장 분석에만 따로 보관 |
| `Type` L/M/H → 0/1/2 | 글자를 숫자로 (품질 등급이라 순서가 있음) |
| 새 특징 3개 | 고장 원리(물리·공학 지식)를 숫자로 표현 |

- **온도차** = 공정 온도 − 공기 온도 → 열이 잘 빠지지 않으면 과열
- **동력(W)** = 토크 × 각속도(= 회전 속도 × 2π/60) → 너무 작거나 크면 전력 이상
- **마모×토크** = 공구 마모 시간 × 토크 → 닳은 공구에 큰 힘이 걸리는 부담(과부하)

In [ ]:
FAIL_TYPES = ['TWF','HDF','PWF','OSF','RNF']
data = df.drop_duplicates().dropna().copy()
fail_type = data[FAIL_TYPES].copy()          # 분석용으로만 보관
data['Type'] = data['Type'].map({'L':0, 'M':1, 'H':2})

def add_features(t):
    t = t.copy()
    t['Temp diff [K]'] = t['Process temperature [K]'] - t['Air temperature [K]']
    t['Power [W]']     = t['Torque [Nm]'] * t['Rotational speed [rpm]'] * 2*np.pi/60
    t['Wear x Torque'] = t['Tool wear [min]'] * t['Torque [Nm]']
    return t

BASE = ['Type'] + num_cols
data = add_features(data)
FEATS = BASE + ['Temp diff [K]', 'Power [W]', 'Wear x Torque']
X, y = data[FEATS], data['Machine failure']
print('입력 특징', len(FEATS), '개:', FEATS)

### 평가용 데이터 떼어 두기
전체의 20%를 **평가용(test)** 으로 떼어 두고 끝까지 손대지 않아요. 모델 고르기와 판단 기준 정하기는 나머지 80%(훈련용) 안에서만 해요.
→ 평가용을 보고 고르면 그 데이터에 맞춘 '컨닝'이 되어 실제보다 성능이 좋게 나오기 때문이에요.

In [ ]:
X_tr, X_te, y_tr, y_te, ft_tr, ft_te = train_test_split(
    X, y, fail_type, test_size=0.2, random_state=SEED, stratify=y)
print('훈련', X_tr.shape, '고장', int(y_tr.sum()), '| 평가', X_te.shape, '고장', int(y_te.sum()))

## 3. 모델 비교 (5겹 층화 교차 검증)
훈련용을 5조각으로 나눠 4조각으로 배우고 1조각으로 평가하기를 5번 돌려요. 한 번만 나눴을 때의 운을 줄일 수 있어요.
`class_weight='balanced'`는 적은 쪽(고장)을 틀리면 벌점을 더 크게 주는 설정이에요.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models = {
    '로지스틱 회귀': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000)),
    '결정 트리':     DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=SEED),
    '랜덤 포레스트': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced',
                                            n_jobs=-1, random_state=SEED),
    '그래디언트 부스팅': HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                                       class_weight='balanced', random_state=SEED),
}
rows, oof = [], {}
base_pred = np.zeros(len(y_tr), dtype=int)
rows.append({'모델': '기준(모두 정상)', **scores(y_tr, base_pred, np.zeros(len(y_tr)))})
for name, m in models.items():
    p = cross_val_predict(m, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
    oof[name] = p
    rows.append({'모델': name, **scores(y_tr, (p >= .5).astype(int), p)})
cv_table = pd.DataFrame(rows).set_index('모델'); cv_table

In [ ]:
BEST = cv_table.drop('기준(모두 정상)')['PR-AUC'].idxmax()
print('PR-AUC 기준 최고 모델:', BEST)
plt.figure(figsize=(6,4.5))
for name, p in oof.items():
    pr, rc, _ = precision_recall_curve(y_tr, p)
    plt.plot(rc, pr, label=f'{name} (AP={average_precision_score(y_tr,p):.2f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall (5-fold CV)'); plt.legend(); plt.show()

➡️ 기준 모델은 정확도는 높지만 재현율이 0이에요. 직선 하나로 나누는 로지스틱 회귀보다 **트리 기반 모델**이 훨씬 좋아요.
고장이 '양 끝'(너무 크거나 작은 동력)에 있는 것처럼, 경계가 직선이 아니기 때문이에요.

### 내가 만든 특징이 정말 도움이 됐을까? (특징 제거 실험)
같은 모델로 ① 원래 특징만 ② 원래 + 새 특징 3개를 교차 검증으로 비교해요.

In [ ]:
abl = []
for label, cols in [('원래 특징 6개', BASE), ('+ 새 특징 3개', FEATS)]:
    p = cross_val_predict(models[BEST], X_tr[cols], y_tr, cv=cv, method='predict_proba')[:, 1]
    abl.append({'특징': label, **scores(y_tr, (p >= .5).astype(int), p)})
pd.DataFrame(abl).set_index('특징')

## 4. 비용 기반 판단 기준(임곗값) 정하기
모델은 '고장 확률'을 내놓고, 기본은 50%가 넘으면 고장이라고 해요. 하지만 공장에서
- 고장을 놓치면(FN) → 라인 정지, 납기 지연, 큰 수리비
- 괜히 점검하면(FP) → 짧은 점검 시간과 인건비

라서 손해가 달라요. **총비용 = 놓친 고장 수 × 10 + 헛경보 수 × 1**(가정)이 가장 작아지는 임곗값을 **교차 검증 결과로** 골라요.

In [ ]:
THRESHOLD, ts, cs = best_threshold(y_tr, oof[BEST])
print(f'{BEST}: 비용 최소 임곗값 = {THRESHOLD}')
plt.figure(figsize=(6,3.6)); plt.plot(ts, cs); plt.axvline(THRESHOLD, c='r', ls='--')
plt.xlabel('threshold'); plt.ylabel('total cost (CV)'); plt.title('Cost vs threshold'); plt.show()

print('비용 비율이 바뀌면 임곗값이 어떻게 달라지나 (민감도 분석)')
for ratio in [1, 5, 10, 20, 50]:
    COST_FN = ratio; t, _, _ = best_threshold(y_tr, oof[BEST])
    pr = (oof[BEST] >= t).astype(int)
    print(f'  FN:FP = {ratio:>2}:1 → 임곗값 {t:.2f} | 재현율 {recall_score(y_tr,pr):.3f} | 정밀도 {precision_score(y_tr,pr):.3f}')
COST_FN = 10

➡️ 고장을 놓치는 손해가 클수록 임곗값이 내려가고 재현율이 올라가요. **판단 기준은 기술만이 아니라 현장의 비용 구조로 정해진다**는 것을 확인했어요.

## 5. 최종 모델 학습과 평가 (평가용 데이터, 딱 한 번)

In [ ]:
final = models[BEST].fit(X_tr, y_tr)
prob_te = final.predict_proba(X_te)[:, 1]
pred_te = (prob_te >= THRESHOLD).astype(int)
res = pd.DataFrame([
    {'모델': '기준(모두 정상)', **scores(y_te, np.zeros(len(y_te),int), np.zeros(len(y_te)))},
    {'모델': f'{BEST} (임곗값 0.5)', **scores(y_te, (prob_te>=.5).astype(int), prob_te)},
    {'모델': f'{BEST} (임곗값 {THRESHOLD})', **scores(y_te, pred_te, prob_te)},
]).set_index('모델'); display(res)
plot_cm(y_te, pred_te, f'{BEST} @ {THRESHOLD}')
print('총비용  모델:', cost(y_te, pred_te), ' / 아무것도 안 할 때:', cost(y_te, np.zeros(len(y_te),int)))

## 6. 결과 해석
### 6-1. 고장 종류별로 얼마나 잡았나
고장 종류 열은 학습에 쓰지 않았지만, 결과를 해석할 때는 쓸 수 있어요.

In [ ]:
rows = []
names = {'TWF':'공구 마모 고장','HDF':'열 방출 고장','PWF':'전력 고장','OSF':'과부하 고장','RNF':'무작위 고장'}
for k in FAIL_TYPES:
    m = ft_te[k] == 1
    if m.sum(): rows.append({'고장 종류': f'{k} ({names[k]})', '평가용 건수': int(m.sum()),
                             '잡은 건수': int(pred_te[m.values].sum()), '재현율': round(pred_te[m.values].mean(), 2)})
pd.DataFrame(rows).set_index('고장 종류')

➡️ 열 방출·전력·과부하 고장처럼 **센서 값의 조합으로 원인이 드러나는 고장은 거의 다 잡지만**,
공구 마모 고장(언제 공구가 부러질지 무작위)이나 무작위 고장은 잘 못 잡아요. **센서 데이터만으로는 예측할 수 없는 고장이 있다**는 한계이자, 공구 교체 주기 같은 **예방 보전 규칙과 함께 써야 하는 이유**예요.

### 6-2. 어떤 특징이 중요했나 (순열 중요도)
특징 하나를 무작위로 섞었을 때 성능(PR-AUC)이 얼마나 떨어지는지로 중요도를 재요. 많이 떨어질수록 중요한 특징이에요.

In [ ]:
pi = permutation_importance(final, X_te, y_te, scoring='average_precision', n_repeats=10, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values()
imp.plot(kind='barh', figsize=(6.5,4), color='seagreen'); plt.title('Permutation importance (PR-AUC drop)'); plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(3))

### 6-3. 데이터 누수 확인 실험
고장 종류 열을 넣으면 어떻게 될까요?

In [ ]:
X_leak = pd.concat([X, fail_type], axis=1)
p = cross_val_predict(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
                      X_leak.loc[X_tr.index], y_tr, cv=cv, method='predict_proba')[:, 1]
print('누수 열을 넣었을 때:', scores(y_tr, (p>=.5).astype(int), p))

➡️ 거의 완벽한 점수가 나와요. 하지만 실제 현장에서는 고장 종류를 고장 **전에** 알 수 없으니 쓸 수 없는 모델이에요. **너무 좋은 결과는 의심해야 한다**는 교훈이에요.

### 6-4. 새 기계 상태로 예측해 보기

In [ ]:
new = pd.DataFrame([[0, 298.1, 308.6, 1551, 42.8, 30],
                    [0, 302.5, 311.0, 1300, 65.0, 210],
                    [2, 300.0, 308.0, 1350, 55.0, 100]],
                   columns=BASE)
new = add_features(new)
for i, v in enumerate(final.predict_proba(new[FEATS])[:, 1]):
    print(f'기계 {i+1}: 고장 확률 {v*100:5.1f}% → {"⚠️ 점검 필요" if v >= THRESHOLD else "✅ 정상 가동"}')

## 7. 외부 검증 A — NASA 항공기 엔진 (C-MAPSS FD001)
**왜?** 수업 데이터 하나에서만 통한 방법이 아닌지 확인하려고, **완전히 다른 기계·다른 센서**의 공개 데이터에 같은 방법(불균형 처리 → 교차 검증 → 비용 기반 임곗값 → 평가용 한 번)을 적용해요.

- 출처: NASA Ames Prognostics Center of Excellence, *Turbofan Engine Degradation Simulation Data Set* (Saxena et al., 2008)
- 엔진 100대가 처음부터 고장 날 때까지의 사이클별 센서 21개 기록 (훈련 20,631행), 평가용 엔진 100대(13,096행)와 실제 남은 수명(RUL)
- **예측할 것**: 이 엔진이 **30사이클 안에 고장 나는가?** (예/아니오 → 분류)
- 주의: 같은 엔진의 기록이 훈련과 검증에 섞이면 컨닝이 되므로 **엔진 단위로** 나눠요(GroupKFold).

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/ramymohamed-sudo/ml-docker/master/scripts/CMAPSSData/'
cols = ['unit','cycle','set1','set2','set3'] + [f's{i}' for i in range(1,22)]
tr = pd.read_csv(BASE_URL+'train_FD001.txt', sep=r'\s+', header=None, names=cols)
te = pd.read_csv(BASE_URL+'test_FD001.txt',  sep=r'\s+', header=None, names=cols)
rul = pd.read_csv(BASE_URL+'RUL_FD001.txt',   sep=r'\s+', header=None, names=['RUL'])
print('훈련', tr.shape, '평가', te.shape, '엔진 수', tr.unit.nunique(), te.unit.nunique())

H = 30   # 30사이클 안에 고장
tr['RUL'] = tr.groupby('unit').cycle.transform('max') - tr.cycle
te['RUL'] = te.unit.map(dict(zip(range(1,101), rul.RUL))) + te.groupby('unit').cycle.transform('max') - te.cycle
for t in (tr, te): t['y'] = (t.RUL <= H).astype(int)

sens = [c for c in cols[5:] if tr[c].std() > 1e-3]      # 거의 안 변하는 센서 제외
def eng(t):   # 특징 만들기: 최근 5사이클 평균(잡음 줄이기)과 변화량(추세)
    t = t.sort_values(['unit','cycle']).copy(); g = t.groupby('unit')
    for c in sens:
        t[c+'_m5'] = g[c].transform(lambda s: s.rolling(5, min_periods=1).mean())
        t[c+'_d5'] = t[c] - g[c].shift(5).fillna(t[c])
    return t
tr, te = eng(tr), eng(te)
FA = ['cycle'] + sens + [c+'_m5' for c in sens] + [c+'_d5' for c in sens]
print('사용 센서', len(sens), '개, 입력 특징', len(FA), '개 / 훈련 고장임박 비율 %.1f%%' % (tr.y.mean()*100))

In [ ]:
mA = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight='balanced', n_jobs=-1, random_state=SEED)
oofA = cross_val_predict(mA, tr[FA], tr.y, cv=GroupKFold(5), groups=tr.unit, method='predict_proba')[:, 1]
thA, _, _ = best_threshold(tr.y, oofA)
mA.fit(tr[FA], tr.y); pA = mA.predict_proba(te[FA])[:, 1]; yA = (pA >= thA).astype(int)
resA = pd.DataFrame([{'데이터':'NASA 엔진 – 기준(모두 정상)', **scores(te.y, np.zeros(len(te),int), np.zeros(len(te)))},
                     {'데이터':f'NASA 엔진 – 랜덤 포레스트 @ {thA}', **scores(te.y, yA, pA)}]).set_index('데이터')
display(resA); plot_cm(te.y, yA, 'NASA C-MAPSS FD001 (test engines)')
impA = pd.Series(mA.feature_importances_, index=FA).sort_values(ascending=False).head(8)
print('중요 특징 상위:', impA.round(3).to_dict())

➡️ 기계도 센서도 다르지만 **같은 방법으로 고장 임박을 높은 재현율로 잡아요.** 중요 특징 상위에 '최근 평균(_m5)'이 많은 것은,
순간 값보다 **추세를 보는 것이 중요하다**는 뜻이에요. 이는 수업 데이터 모델의 한계(한 시점만 봄)를 보완할 방향이에요.

## 8. 외부 검증 B — Microsoft Azure 설비 예지보전 데이터
- 출처: Microsoft *Azure AI Predictive Maintenance* 예제 데이터 (기계 100대, 2015년 1년치 1시간 간격 센서·오류·정비·고장 기록)
- **예측할 것**: 이 기계가 **앞으로 24시간 안에 고장 나는가?**
- **시간 순서로 나눔**: 1~8월 학습, 9월로 임곗값 결정, 10~12월 평가 → '과거로 배워 미래를 예측'하는 실제 상황과 똑같이
- 산업공학 관점 특징: 최근 24시간 센서 평균·변동, 최근 오류 횟수, **부품별 마지막 교체 후 지난 시간**(정비 이력), 기계 나이
- 데이터가 커서(약 88만 행) 불러오는 데 1~2분 걸려요.

In [ ]:
MS = 'https://media.githubusercontent.com/media/aryawiandra/asset-health-monitoring/main/data/raw/'
RAW = 'https://raw.githubusercontent.com/aryawiandra/asset-health-monitoring/main/data/raw/'
tel  = pd.read_csv(MS+'PdM_telemetry.csv', parse_dates=['datetime'])
err  = pd.read_csv(RAW+'PdM_errors.csv',   parse_dates=['datetime'])
mnt  = pd.read_csv(RAW+'PdM_maint.csv',    parse_dates=['datetime'])
fai  = pd.read_csv(RAW+'PdM_failures.csv', parse_dates=['datetime'])
mac  = pd.read_csv(RAW+'PdM_machines.csv')
print('센서', tel.shape, '오류', err.shape, '정비', mnt.shape, '고장', fai.shape, '기계', mac.shape)

In [ ]:
S4 = ['volt','rotate','pressure','vibration']
tel = tel.sort_values(['machineID','datetime']).reset_index(drop=True)
g = tel.groupby('machineID')
for c in S4:
    tel[c+'_mean24'] = g[c].transform(lambda s: s.rolling(24, min_periods=1).mean())
    tel[c+'_std24']  = g[c].transform(lambda s: s.rolling(24, min_periods=2).std()).fillna(0)
# 최근 24시간 오류 횟수 (오류 종류별)
e = pd.get_dummies(err.set_index(['machineID','datetime'])['errorID']).groupby(level=[0,1]).sum().reset_index()
tel = tel.merge(e, on=['machineID','datetime'], how='left'); ER = [c for c in e.columns if c.startswith('error')]
tel[ER] = tel[ER].fillna(0)
for c in ER: tel[c+'_24h'] = tel.groupby('machineID')[c].transform(lambda s: s.rolling(24, min_periods=1).sum())
# 부품별 마지막 교체 후 지난 시간(일)
for comp in ['comp1','comp2','comp3','comp4']:
    m = mnt[mnt.comp==comp][['machineID','datetime']].rename(columns={'datetime':'last_'+comp}).sort_values('last_'+comp)
    tel = pd.merge_asof(tel.sort_values('datetime'), m, left_on='datetime', right_on='last_'+comp, by='machineID')
    tel['days_since_'+comp] = ((tel.datetime - tel['last_'+comp]).dt.total_seconds()/86400).fillna(365)
tel = tel.merge(mac, on='machineID'); tel['model'] = tel.model.str[-1].astype(int)
# 정답: 앞으로 24시간 안에 고장
f = fai.drop_duplicates(['machineID','datetime'])[['machineID','datetime']].rename(columns={'datetime':'fail_time'}).sort_values('fail_time')
tel = pd.merge_asof(tel.sort_values('datetime'), f, left_on='datetime', right_on='fail_time', by='machineID', direction='forward')
tel['y'] = ((tel.fail_time - tel.datetime).dt.total_seconds().between(0, 24*3600)).astype(int)
tel = tel[tel.datetime.dt.hour % 3 == 0]     # 3시간 간격만 사용(속도)
FB = [c+s for c in S4 for s in ['', '_mean24', '_std24']] + [c+'_24h' for c in ER] + \
     [f'days_since_comp{i}' for i in range(1,5)] + ['age','model']
print('행', len(tel), '특징', len(FB), '고장임박 비율 %.2f%%' % (tel.y.mean()*100))

In [ ]:
trB = tel[tel.datetime <  '2015-09-01']; vaB = tel[(tel.datetime >= '2015-09-01') & (tel.datetime < '2015-10-01')]
teB = tel[tel.datetime >= '2015-10-01']
mB = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced', n_jobs=-1, random_state=SEED)
mB.fit(trB[FB], trB.y)
thB, _, _ = best_threshold(vaB.y, mB.predict_proba(vaB[FB])[:, 1])
mB.fit(pd.concat([trB, vaB])[FB], pd.concat([trB, vaB]).y)    # 임곗값을 정한 뒤 9월까지 다시 학습
pB = mB.predict_proba(teB[FB])[:, 1]; yB = (pB >= thB).astype(int)
resB = pd.DataFrame([{'데이터':'Microsoft 설비 – 기준(모두 정상)', **scores(teB.y, np.zeros(len(teB),int), np.zeros(len(teB)))},
                     {'데이터':f'Microsoft 설비 – 랜덤 포레스트 @ {thB}', **scores(teB.y, yB, pB)}]).set_index('데이터')
display(resB); plot_cm(teB.y, yB, 'Microsoft Azure PdM (Oct-Dec 2015)')
impB = pd.Series(mB.feature_importances_, index=FB).sort_values(ascending=False).head(8)
print('중요 특징 상위:', impB.round(3).to_dict())

➡️ 미래(10~12월)를 예측하는 엄격한 조건에서도 고장 임박을 잡아냈어요. 중요 특징에 **센서 24시간 평균**과 **오류 횟수**, **부품 교체 후 지난 시간**이 올라온 것은
'센서 추세 + 정비 이력'을 함께 보는 것이 실제 설비 관리에 중요하다는 뜻이에요.

## 9. 종합 결론 — 세 데이터에서 같은 결론이 나왔나?

In [ ]:
summary = pd.concat([res, resA, resB]); summary

| 확인한 것 | 기계 고장(수업) | NASA 엔진 | Microsoft 설비 |
|---|---|---|---|
| 정확도만 보면 속는다(기준 모델 재현율 0) | ✅ | ✅ | ✅ |
| 불균형 처리 + 트리 모델로 고장을 높은 재현율로 잡음 | ✅ | ✅ | ✅ |
| 비용 기반 임곗값이 재현율을 끌어올림 | ✅ | ✅ | ✅ |
| 공학적 의미가 있는 특징이 중요 특징으로 나옴 | 동력·마모×토크 | 센서 최근 평균(추세) | 센서 24h 평균·오류·교체 후 시간 |

**결론**: 이 방법은 한 데이터에서만 우연히 통한 것이 아니라 **다른 기계·다른 센서·다른 시간 구조에서도 재현**돼요.
**한계와 다음 단계**: ① 무작위 고장·공구 파손은 센서로 예측하기 어려워 예방 보전 규칙과 함께 써야 함 ② 수업 데이터는 한 시점만 보므로, 외부 검증에서 확인한 것처럼 **시간 추세 특징**을 넣으면 더 좋아질 것 ③ 비용 비율(10:1)은 가정이라 실제 공장의 고장·점검 비용 자료가 필요함.